## LLamaParse

**1. Parsing Document** \
**2. Parsing financial Data** \
**3. Extracting Relevant Data from PDF** \
**4. Classify Documents**

### Installation

```bash
pip install llama-cloud>=2.1
```

- **Register @ llamaindex.ai:**

- **Generate and Copy your the API Key**

In [ ]:
import os

from llama_cloud import LlamaCloud
from dotenv import load_dotenv

load_dotenv()

os.environ['LLAMA_CLOUD_API_KEY'] = os.getenv("LLAMA_CLOUD_API_KEY")

### Parse Document

In [ ]:
client = LlamaCloud()  # reads LLAMA_CLOUD_API_KEY from the environment

# Upload and parse a document
file = client.files.create(file="./LLM-Interview-Questions.pdf", purpose="parse")
result = client.parsing.parse(
    file_id=file.id,
    tier="agentic",
    version="latest",
    expand=["markdown"],
)

# Print the markdown for the first page
print(result.markdown.pages[0].markdown)

### Parsing Table from Document

In [ ]:
from pydantic import config
from llama_cloud import AsyncLlamaCloud

client = AsyncLlamaCloud()

file_obj = await client.files.create(
    file="MonthlyReceipt.pdf",
    purpose="parse",
    
)

result = await client.parsing.parse(
    file_id=file_obj.id,
    tier="agentic",
    version="latest",
    page_ranges= {'target_pages': '17-18'},
    processing_options={
        "cost_optimizer": {"enable": True},
    },
    expand=["markdown", "items", "metadata"],

)

print(f"Job status: {result.job.status}")
print(f"Total pages: {len(result.items.pages)}")

In [ ]:
for page in result.metadata.pages:
    flag = "cost-optimized" if page.cost_optimized else "premium tier"
    print(f"page {page.page_number}: {flag}")

In [ ]:
all_tables = []

for page in result.items.pages:
    for item in page.items:
        if getattr(item, "type", None) == "table":
            all_tables.append({
                "page_number": page.page_number,
                "rows": item.rows,
                "csv": item.csv,
            })

print(f"Found {len(all_tables)} tables across {len(result.items.pages)} pages.")
for t in all_tables:
    n_rows = len(t["rows"])
    n_cols = len(t["rows"][0]) if t["rows"] else 0
    print(f"  page {t['page_number']}: {n_rows} rows × {n_cols} cols")

In [ ]:
import io
import pandas as pd

dataframes = []
for t in all_tables:
    df = pd.read_csv(io.StringIO(t["csv"]))
    df["_source_page"] = t["page_number"]  # keep the source page for traceability
    dataframes.append(df)

# Quick look at the first table
if dataframes:
    print(dataframes[0].head())
    print(f"\nColumns: {list(dataframes[0].columns)}")

In [ ]:
# Tables with at least 5 rows (skip tiny one-line summaries)
substantial_tables = [df for df in dataframes if len(df) >= 5]

# Or filter by column header contents
financial_tables = [
    df for df in dataframes
    if any("Cash" in str(c) or "$" in str(c) for c in df.columns)
]

print(f"Substantial tables: {len(substantial_tables)}")
print(f"Financial tables (with $ in column headers): {len(financial_tables)}")

In [ ]:
# Save each table as a separate CSV with a deterministic filename
for i, t in enumerate(all_tables):
    filename = f"table_p{t['page_number']:03d}_{i}.csv"
    pd.read_csv(io.StringIO(t["csv"])).to_csv(filename, index=False)

# Or write them all into a single Excel workbook, one sheet per table
with pd.ExcelWriter("financial_tables.xlsx") as writer:
    for i, t in enumerate(all_tables):
        df = pd.read_csv(io.StringIO(t["csv"]))
        sheet_name = f"page{t['page_number']}_{i}"[:31]  # Excel limit
        df.to_excel(writer, sheet_name=sheet_name, index=False)

## Extract

In [ ]:
import time
from pydantic import BaseModel, Field
from llama_cloud import LlamaCloud

# Define schema using Pydantic
class Questions(BaseModel):
    question: str = Field(description="Extarct LLM Questions")

# Extract data from document
question = client.extract.create(
    file_input=file.id,
    configuration={
        "data_schema": Questions.model_json_schema(),
        "extraction_target": "per_doc",
        "tier": "agentic",
    },
)

In [ ]:
# Poll for completion
while question.status not in ("COMPLETED", "FAILED", "CANCELLED"):
    time.sleep(2)
    question = client.extract.get(question.id)

print(question.extract_result)

In [ ]:
print(question.extract_result)

### Classify

In [ ]:
import os
from llama_cloud import LlamaCloud, AsyncLlamaCloud

# Upload a file
file_obj = client.files.create(file="./ml_engineer.pdf", purpose="classify")
file_id = file_obj.id

# Classify and wait for completion
result = client.classifier.classify(
    file_ids=[file_id],
    rules=[
        {
            "type": "resume",
            "description": "Documents that contain an name, qualification, skills, and experience.",
        },
        {
            "type": "receipt",
            "description": "Short purchase receipts, typically from POS systems, with merchant, items and total, often a single page.",
        },
    ],
    parsing_configuration={
        "lang": "en",
        "max_pages": 1,            # optional, parse at most 5 pages
        # "target_pages": "1,3",   # optional, parse only specific pages (1-based)
    },
    mode="FAST",  # or "MULTIMODAL"
)

# Print the classification results
for item in result.items:
    if item.result is None:
        print(f"Classification failed for file {item.file_id}")
        continue
    print(f"Classified type: {item.result.type}")
    print(f"Confidence: {item.result.confidence}")
    print(f"Reasoning: {item.result.reasoning}")